In [ ]:
import requests
import json
import logging
import pandas as pd

In [ ]:
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [3]:
logging.basicConfig(
    level=logging.INFO,
    filename="project.log",
    format="%(asctime)s - %(module)s - %(levelname)s - %(funcName)s: %(lineno)d - %(message)s",
    datefmt="%H:%M:%S"
)

logging.info("Логирование настроено")
logging.info("тест")

In [18]:
with open("project.log", "r", encoding="utf-8") as f:
    print(f.read())

18:58:09 - 3579905043 - INFO - <cell line: 0>: 8 - Логирование настроено
18:58:09 - 3579905043 - INFO - <cell line: 0>: 9 - тест
18:58:15 - 2455892144 - INFO - <cell line: 0>: 1 - тест
18:58:18 - 4222666904 - INFO - <cell line: 0>: 2 - Ключ получен
18:58:21 - 2537822844 - INFO - <cell line: 0>: 8 - Получили датафрейм городов для работы и оставили только те, где есть аэропорт
18:58:25 - 2960473042 - INFO - <cell line: 0>: 1 - загружаем справочник городов у Travelpayouts, чтобы потом сопоставить наши города с кодами, которые нужны дальше для запросов по авиабилетам
18:58:25 - 2960473042 - INFO - <cell line: 0>: 5 - Справочник городов загружен
18:58:28 - 1903839447 - INFO - <cell line: 0>: 3 - Таблица городов из API собрана
18:58:32 - 2514882744 - INFO - <cell line: 0>: 10 - IATA-коды добавлены в таблицу городов
19:01:47 - 3410942844 - INFO - <cell line: 0>: 5 - собираем цены для 3 городов
19:01:47 - 3410942844 - INFO - <cell line: 0>: 36 - Нашли цену для Абакан to Анадырь
19:01:48 - 3410

In [5]:
logging.info("тест")

In [ ]:
API_key = "XXXXXXXXXXXXXXXXXXXXXXXXX"
logging.info("Ключ получен")

In [7]:
df = pd.read_excel("Итоговый список городов-2.xlsx")

df = df[["Город", "Регион", "Население", "Наличие аэропорта"]].copy()
df = df[df["Наличие аэропорта"] == True]

print(df.head())
print(df.shape)
logging.info("Получили датафрейм городов для работы и оставили только те, где есть аэропорт")

         Город                 Регион  Население  Наличие аэропорта
0       Абакан                Хакасия     184284               True
1      Анадырь           Чукотский АО      13224               True
2        Анапа     Краснодарский край      84804               True
4  Архангельск  Архангельская область     294914               True
5    Астрахань   Астраханская область     465990               True
(75, 4)


In [8]:
logging.info("загружаем справочник городов у Travelpayouts, чтобы потом сопоставить наши города с кодами, которые нужны дальше для запросов по авиабилетам")

page = requests.get("http://api.travelpayouts.com/data/ru/cities.json")
if page.status_code == 200:
    logging.info("Справочник городов загружен")
else:
    logging.warning(f"упало с кодом={page.status_code}")


In [9]:
cities_data = page.json()
cities_df = pd.DataFrame(cities_data)
logging.info("Таблица городов из API собрана")
cities_df.head()

,name_translations,cases,country_code,code,time_zone,name,coordinates,has_flightable_airport
0,{'en': 'Uruguaiana'},"{'da': 'Уругуаяне', 'pr': 'Уругуаяне', 'ro': '...",BR,URG,America/Sao_Paulo,Уругуаяна,"{'lat': -29.781668, 'lon': -57.038334}",True
1,{'en': 'Kimberley Downs'},"{'da': 'Кимберли-Даунс', 'pr': 'Кимберли-Даунс...",AU,KBD,Australia/Perth,Кимберли-Даунс,"{'lat': -17.333332, 'lon': 124.35}",False
2,{'en': 'Palm Island'},"{'da': 'Палм-Айленду', 'pr': 'Палм-Айленде', '...",VC,PLI,America/St_Vincent,Палм-Айленд,"{'lat': 12.35, 'lon': -61.233334}",False
3,{'en': 'Navoi'},"{'da': 'Навои', 'pr': 'Навои', 'ro': 'Навои', ...",UZ,NVI,Asia/Samarkand,Навои,"{'lat': 40.115, 'lon': 65.159164}",True
4,{'en': 'Sundsvall'},"{'da': 'Сундсваллю', 'pr': 'Сундсвалле', 'ro':...",SE,SDL,Europe/Stockholm,Сундсвалль,"{'lat': 62.3908358, 'lon': 17.3069157}",True


In [10]:
df = df.merge(
    cities_df,
    left_on="Город",
    right_on="name",
    how="left"
)

df = df.rename(columns={"code": "iata_code"})
df = df.drop(columns=["name"])
logging.info("IATA-коды добавлены в таблицу городов")
df

,Город,Регион,Население,Наличие аэропорта,name_translations,cases,country_code,iata_code,time_zone,coordinates,has_flightable_airport
0,Абакан,Хакасия,184284,True,{'en': 'Abakan'},"{'da': 'Абакану', 'pr': 'Абакане', 'ro': 'Абак...",RU,ABA,Asia/Krasnoyarsk,"{'lat': 53.716667, 'lon': 91.5}",True
1,Анадырь,Чукотский АО,13224,True,{'en': 'Anadyr'},"{'da': 'Анадырю', 'pr': 'Анадыре', 'ro': 'Анад...",RU,DYR,Asia/Anadyr,"{'lat': 64.73333, 'lon': 177.75}",True
2,Анапа,Краснодарский край,84804,True,{'en': 'Anapa'},"{'da': 'Анапе', 'pr': 'Анапе', 'ro': 'Анапы', ...",RU,AAQ,Europe/Moscow,"{'lat': 44.9, 'lon': 37.316666}",True
3,Архангельск,Архангельская область,294914,True,{'en': 'Arkhangelsk'},"{'da': 'Архангельску', 'pr': 'Архангельске', '...",RU,ARH,Europe/Moscow,"{'lat': 64.594795, 'lon': 40.711903}",True
4,Астрахань,Астраханская область,465990,True,{'en': 'Astrakhan'},"{'da': 'Астрахани', 'pr': 'Астрахани', 'ro': '...",RU,ASF,Europe/Samara,"{'lat': 46.2877, 'lon': 47.999977}",True
...,...,...,...,...,...,...,...,...,...,...,...
70,Чита,Забайкальский край,337063,True,{'en': 'Chita'},"{'da': 'Чите', 'pr': 'Чите', 'ro': 'Читы', 'su...",RU,HTA,Asia/Chita,"{'lat': 52.033333, 'lon': 113.3}",True
71,Элиста,Калмыкия,104082,True,{'en': 'Elista'},"{'da': 'Элисте', 'pr': 'Элисте', 'ro': 'Элисты...",RU,ESL,Europe/Moscow,"{'lat': 46.36667, 'lon': 44.333332}",True
72,Южно-Сахалинск,Сахалинская область,181976,True,{'en': 'Yuzhno-Sakhalinsk'},"{'da': 'Южно-Сахалинску', 'pr': 'Южно-Сахалинс...",RU,UUS,Asia/Srednekolymsk,"{'lat': 46.966667, 'lon': 142.75}",True
73,Якутск,Якутия,372801,True,{'en': 'Yakutsk'},"{'da': 'Якутску', 'pr': 'Якутске', 'ro': 'Якут...",RU,YKS,Asia/Yakutsk,"{'lat': 62.085606, 'lon': 129.75006}",True


In [11]:
df_target = df[["Город", "Регион", "Население", "Наличие аэропорта", "country_code", "iata_code"]].copy()
df_target

,Город,Регион,Население,Наличие аэропорта,country_code,iata_code
0,Абакан,Хакасия,184284,True,RU,ABA
1,Анадырь,Чукотский АО,13224,True,RU,DYR
2,Анапа,Краснодарский край,84804,True,RU,AAQ
3,Архангельск,Архангельская область,294914,True,RU,ARH
4,Астрахань,Астраханская область,465990,True,RU,ASF
...,...,...,...,...,...,...
70,Чита,Забайкальский край,337063,True,RU,HTA
71,Элиста,Калмыкия,104082,True,RU,ESL
72,Южно-Сахалинск,Сахалинская область,181976,True,RU,UUS
73,Якутск,Якутия,372801,True,RU,YKS


In [12]:
print(df_target["iata_code"].isna().sum())

2


In [13]:
df_target = df_target[df_target["iata_code"].notna()].reset_index(drop=True)

In [14]:
print(df_target["iata_code"].isna().sum())

0


Делаем запросы на тестовой выборке

In [15]:
test_cities = df_target.head(3).reset_index(drop=True)

flights_df = pd.DataFrame()

logging.info(f"собираем цены для {len(test_cities)} городов")

for i in range(len(test_cities)):
    for j in range(len(test_cities)):

        from_city = test_cities.loc[i, "Город"]
        from_code = test_cities.loc[i, "iata_code"]

        to_city = test_cities.loc[j, "Город"]
        to_code = test_cities.loc[j, "iata_code"]

        if from_city == to_city:
            continue

        # https://support.travelpayouts.com/hc/ru/articles/203956163-API-%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85-Aviasales-%D0%B4%D0%BB%D1%8F-%D0%BF%D0%B0%D1%80%D1%82%D0%BD%D1%91%D1%80%D0%BE%D0%B2
        request = f"https://api.travelpayouts.com/aviasales/v3/prices_for_dates?origin={from_code}&destination={to_code}&departure_at=2026-06&one_way=true&direct=false&currency=rub&sorting=price&limit=1&page=1"


        page = requests.get(request, headers={"accept": "application/json", "X-Access-Token": API_key})

        ans = page.json()

        if len(ans["data"]) > 0:
            page = pd.DataFrame(ans["data"])

            page["from_city"] = from_city
            page["from_code"] = from_code
            page["to_city"] = to_city
            page["to_code"] = to_code

            page = page[["from_city", "from_code", "to_city", "to_code", "price", "duration"]]

            flights_df = pd.concat([flights_df, page], ignore_index=True)
            logging.info(f"Нашли цену для {from_city} to {to_city}")

        else:
            page = pd.DataFrame([{"from_city": from_city, "from_code": from_code, "to_city": to_city, "to_code": to_code, "price": None, "duration": None}])

            flights_df = pd.concat([flights_df, page], ignore_index=True)
            logging.info(f"Нет данных для маршрута {from_city} to {to_city}")

logging.info(f"все готово, получилось строк: {len(flights_df)}")

flights_df

,from_city,from_code,to_city,to_code,price,duration
0,Абакан,ABA,Анадырь,DYR,39655,2850
1,Абакан,ABA,Анапа,AAQ,None,None
2,Анадырь,DYR,Абакан,ABA,None,None
3,Анадырь,DYR,Анапа,AAQ,None,None
4,Анапа,AAQ,Абакан,ABA,None,None
5,Анапа,AAQ,Анадырь,DYR,None,None


Смотрим теперь на всех парах городов

In [17]:
test_cities = df_target.reset_index(drop=True)

flights_df = pd.DataFrame()

logging.info(f"собираем цены для {len(test_cities)} городов")

for i in range(len(test_cities)):
    for j in range(len(test_cities)):

        from_city = test_cities.loc[i, "Город"]
        from_code = test_cities.loc[i, "iata_code"]

        to_city = test_cities.loc[j, "Город"]
        to_code = test_cities.loc[j, "iata_code"]

        if from_city == to_city:
            continue

        # https://support.travelpayouts.com/hc/ru/articles/203956163-API-%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85-Aviasales-%D0%B4%D0%BB%D1%8F-%D0%BF%D0%B0%D1%80%D1%82%D0%BD%D1%91%D1%80%D0%BE%D0%B2
        request = f"https://api.travelpayouts.com/aviasales/v3/prices_for_dates?origin={from_code}&destination={to_code}&departure_at=2026-06&one_way=true&direct=false&currency=rub&sorting=price&limit=1&page=1"


        page = requests.get(request, headers={"accept": "application/json", "X-Access-Token": API_key})

        ans = page.json()

        if len(ans["data"]) > 0:
            page = pd.DataFrame(ans["data"])

            page["from_city"] = from_city
            page["from_code"] = from_code
            page["to_city"] = to_city
            page["to_code"] = to_code

            page = page[["from_city", "from_code", "to_city", "to_code", "price", "duration"]]

            flights_df = pd.concat([flights_df, page], ignore_index=True)
            logging.info(f"Нашли цену для {from_city} to {to_city}")

        else:
            page = pd.DataFrame([{"from_city": from_city, "from_code": from_code, "to_city": to_city, "to_code": to_code, "price": None, "duration": None}])

            flights_df = pd.concat([flights_df, page], ignore_index=True)
            logging.info(f"Нет данных для маршрута {from_city} to {to_city}")



logging.info(f"все готово, получилось строк: {len(flights_df)}")

flights_df

,from_city,from_code,to_city,to_code,price,duration
0,Абакан,ABA,Анадырь,DYR,39655,2850
1,Абакан,ABA,Анапа,AAQ,None,None
2,Абакан,ABA,Архангельск,ARH,None,None
3,Абакан,ABA,Астрахань,ASF,None,None
4,Абакан,ABA,Барнаул,BAX,11714,175
...,...,...,...,...,...,...
5251,Ярославль,IAR,Череповец,CEE,None,None
5252,Ярославль,IAR,Чита,HTA,None,None
5253,Ярославль,IAR,Элиста,ESL,None,None
5254,Ярославль,IAR,Южно-Сахалинск,UUS,None,None


In [19]:
flights = flights_df[flights_df["price"].notna()]
flights

,from_city,from_code,to_city,to_code,price,duration
0,Абакан,ABA,Анадырь,DYR,39655,2850
4,Абакан,ABA,Барнаул,BAX,11714,175
9,Абакан,ABA,Владивосток,VVO,20241,2015
10,Абакан,ABA,Владикавказ,OGZ,27882,1900
11,Абакан,ABA,Волгоград,VOG,19657,850
...,...,...,...,...,...,...
5232,Ярославль,IAR,Самара,KUF,13557,1035
5233,Ярославль,IAR,Санкт-Петербург,LED,4718,95
5237,Ярославль,IAR,Сочи,AER,9480,215
5240,Ярославль,IAR,Сыктывкар,SCW,10702,1460


In [20]:
flights.to_excel("flights.xlsx", index=False)

In [22]:
from google.colab import files
files.download("flights.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from google.colab import files
files.download("project.log")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>